# E02 — replier BatchNorm dans la couche Linear

En inférence, BatchNorm utilise des stats fixes : c'est une transformation affine constante.
On peut donc l'absorber dans la couche Linear qui la précède, puis la supprimer.

Pour une couche `Linear` sans biais suivie de BatchNorm, colonne par colonne :

$$y = \gamma \frac{xW - \mu}{\sigma} + \beta = x \left( W \frac{\gamma}{\sigma} \right) + \left( \beta - \frac{\gamma \mu}{\sigma} \right)$$

D'où :

- `W_f = W * (gain / std)` : chaque colonne `j` de `W` est multipliée par `gain[j] / std[j]`
- `b_f = bias - gain * mean / std`

Modèle : MLP à 3 couches Linear, avec BatchNorm après les deux premières.

In [1]:
import random
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s : i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}
vocab_size = len(itos)

random.seed(42)
ws = words[:]
random.shuffle(ws)
n1, n2 = int(0.8 * len(ws)), int(0.9 * len(ws))

block_size = 3

def build_dataset(words):
	X, Y = [], []
	for w in words:
		context = [0] * block_size
		for ch in w + '.':
			ix = stoi[ch]
			X.append(context)
			Y.append(ix)
			context = context[1:] + [ix]
	X = torch.tensor(X)
	Y = torch.tensor(Y)
	print(X.shape, Y.shape)
	return X, Y

Xtr, Ytr = build_dataset(ws[:n1])
Xdev, Ydev = build_dataset(ws[n1:n2])
Xte, Yte = build_dataset(ws[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [2]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / ((block_size * n_embd)**0.5)
W2 = torch.randn((n_hidden, n_hidden), generator=g) * (5/3) / (n_hidden**0.5)
W3 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b3 = torch.randn(vocab_size, generator=g) * 0

bngain1 = torch.ones((1, n_hidden))
bnbias1 = torch.zeros((1, n_hidden))
bnmean_running1 = torch.zeros((1, n_hidden))
bnstd_running1 = torch.ones((1, n_hidden))

bngain2 = torch.ones((1, n_hidden))
bnbias2 = torch.zeros((1, n_hidden))
bnmean_running2 = torch.zeros((1, n_hidden))
bnstd_running2 = torch.ones((1, n_hidden))

parameters = [C, W1, W2, W3, b3, bngain1, bnbias1, bngain2, bnbias2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
	p.requires_grad = True

52497


In [3]:
max_steps = 200000
batch_size = 64
lossi = []

for i in range(max_steps):
	ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=g)
	Xb, Yb = Xtr[ix], Ytr[ix]

	#forward pass
	emb = C[Xb]
	embcat = emb.view(emb.shape[0], -1)

	# couche 1
	hpreact1 = embcat @ W1
	bnmeani1 = hpreact1.mean(0, keepdim=True)
	bnstdi1 = hpreact1.std(0, keepdim=True)
	hpreact1 = bngain1 * (hpreact1 - bnmeani1) / bnstdi1 + bnbias1
	h1 = torch.tanh(hpreact1)

	# couche 2
	hpreact2 = h1 @ W2
	bnmeani2 = hpreact2.mean(0, keepdim=True)
	bnstdi2 = hpreact2.std(0, keepdim=True)
	hpreact2 = bngain2 * (hpreact2 - bnmeani2) / bnstdi2 + bnbias2
	h2 = torch.tanh(hpreact2)

	with torch.no_grad():
		bnmean_running1 = 0.999 * bnmean_running1 + 0.001 * bnmeani1
		bnstd_running1 = 0.999 * bnstd_running1 + 0.001 * bnstdi1
		bnmean_running2 = 0.999 * bnmean_running2 + 0.001 * bnmeani2
		bnstd_running2 = 0.999 * bnstd_running2 + 0.001 * bnstdi2

	logits = h2 @ W3 + b3
	loss = F.cross_entropy(logits, Yb)

	#backward pass
	for p in parameters:
		p.grad = None
	loss.backward()

	#update
	lr = 0.1 if i < 100000 else 0.01
	for p in parameters:
		p.data -= lr * p.grad

	lossi.append(loss.log10().item())
	if i % 10000 == 0:
		print(f"{i:7d}/{max_steps:7d} : {loss.item():.4f}")

      0/ 200000 : 3.3022
  10000/ 200000 : 2.3165
  20000/ 200000 : 2.2565
  30000/ 200000 : 2.0715
  40000/ 200000 : 1.8125
  50000/ 200000 : 1.9626
  60000/ 200000 : 2.2649
  70000/ 200000 : 2.0680
  80000/ 200000 : 2.0245
  90000/ 200000 : 1.9139
 100000/ 200000 : 1.9595
 110000/ 200000 : 1.9554
 120000/ 200000 : 1.9544
 130000/ 200000 : 1.7223
 140000/ 200000 : 2.0724
 150000/ 200000 : 1.9117
 160000/ 200000 : 2.0808
 170000/ 200000 : 2.1409
 180000/ 200000 : 2.4074
 190000/ 200000 : 1.9365


## Deux forwards d'inférence

In [4]:
@torch.no_grad()
def forward_bn(x):
	emb = C[x]
	embcat = emb.view(emb.shape[0], -1)
	h1 = torch.tanh(bngain1 * (embcat @ W1 - bnmean_running1) / bnstd_running1 + bnbias1)
	h2 = torch.tanh(bngain2 * (h1 @ W2 - bnmean_running2) / bnstd_running2 + bnbias2)
	return h2 @ W3 + b3

# repli : une seule fois, après l'entraînement
with torch.no_grad():
	W1f = W1 * (bngain1 / bnstd_running1)
	b1f = bnbias1 - bngain1 * bnmean_running1 / bnstd_running1
	W2f = W2 * (bngain2 / bnstd_running2)
	b2f = bnbias2 - bngain2 * bnmean_running2 / bnstd_running2

@torch.no_grad()
def forward_folded(x):
	emb = C[x]
	embcat = emb.view(emb.shape[0], -1)
	h1 = torch.tanh(embcat @ W1f + b1f)
	h2 = torch.tanh(h1 @ W2f + b2f)
	return h2 @ W3 + b3

In [5]:
for split, (x, y) in {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev)}.items():
	lb = forward_bn(x)
	lf = forward_folded(x)
	print(f"{split:>5} | allclose : {torch.allclose(lb, lf, atol=1e-5)} | écart max : {(lb - lf).abs().max().item():.2e}"
		  f" | loss bn : {F.cross_entropy(lb, y).item():.4f} | loss folded : {F.cross_entropy(lf, y).item():.4f}")

train | allclose : True | écart max : 4.77e-06 | loss bn : 1.9798 | loss folded : 1.9798
  dev | allclose : True | écart max : 4.77e-06 | loss bn : 2.0735 | loss folded : 2.0735


## Échantillonnage avec le modèle replié

In [6]:
g_s = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
	out = []
	context = [0] * block_size
	while True:
		probs = F.softmax(forward_folded(torch.tensor([context])), dim=1)
		ix = torch.multinomial(probs, num_samples=1, generator=g_s).item()
		context = context[1:] + [ix]
		if ix == 0:
			break
		out.append(itos[ix])
	print(''.join(out))

carmahzabelle
khyrir
xith
cassanden
jazonte
deliah
jareen
nellara
chaily
kaleigh
ham
port
quintis
lilah
jadis
wavero
dearyxir
kael
dustine
deci


## Conclusion

Le modèle replié donne les mêmes logits que le modèle avec BatchNorm, à la précision float32 près.
BatchNorm ne sert donc qu'à stabiliser l'entraînement : une fois les stats figées, il ne contient aucune
capacité que la couche Linear ne puisse représenter seule. En production, on le replie et on économise
une normalisation par couche à chaque inférence.

Ce repli ne marche que parce que la normalisation dépend de stats **fixes** en inférence.
LayerNorm calcule ses stats sur chaque exemple, au moment de l'inférence : on ne peut pas la replier.